# Diabetes Prediction System — Custom Machine Learning Model

---

## Project Overview

This notebook analyzes the **Pima Indians Diabetes Database** to predict whether a patient has diabetes based on diagnostic measurements.  
The dataset covers 768 female patients of Pima Indian heritage and includes 9 features such as glucose levels, BMI, insulin, and age. The core of this project is a **custom-built Logistic Regression algorithm** created from scratch to perform the classification.

### Objectives
- Perform exploratory data analysis (EDA) on the diabetes dataset
- Standardize the medical features using `StandardScaler`
- Build a custom Logistic Regression model using Gradient Descent without relying on pre-built ML model libraries
- Evaluate model accuracy on training and testing data
- Create a predictive system for individual patient diagnosis

### Dataset Details
| Attribute | Value |
|-----------|-------|
| Source | `diabetes.csv` |
| Rows | 768 |
| Columns | 9 |
| Target Variable | `Outcome` (0 = Non-Diabetic, 1 = Diabetic) |

---
## 1. Importing Libraries

We start by importing the essential Python libraries for data manipulation, mathematical operations, and preprocessing:
- **NumPy** — numerical operations and array handling
- **Pandas** — data loading and analysis
- **Scikit-Learn** — data standardization and splitting

In [ ]:
# Importing the required libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

---
## 2. Loading the Dataset

The dataset is loaded from a local CSV file into a Pandas DataFrame.

In [ ]:
# Reading the csv file
diabities_dataset = pd.read_csv('diabetes.csv')

---
## 3. Exploratory Data Analysis (EDA)

We begin by getting a quick feel for the data — inspecting the rows, the shape, and the statistical summary.

### 3.1 First 5 Rows (`head`)

A quick look at the top of the dataset to understand its structure and content.

In [ ]:
# Top five rows
print("The top five rows are: ")
diabities_dataset.head()

### 3.2 Dataset Shape

The shape tells us the total number of records (rows) and features (columns) in the dataset.

In [ ]:
# Shape of the dataset in the format of (rows, columns)
print("The shape is: ")
diabities_dataset.shape

### 3.3 Statistical Summary (`describe`)

The `describe()` function provides key statistical measures — mean, std, min, max, and quartiles — for our medical data.

In [ ]:
diabities_dataset.describe()

---
## 4. Data Preprocessing & Standardization

Machine learning models perform better when numerical input variables are scaled to a standard range. Here we separate the features and standardize them.

In [ ]:
# Separate features and target
features = diabities_dataset.drop(columns='Outcome')
target = diabities_dataset['Outcome']

In [ ]:
# Standardize the data using StandardScaler
scaler = StandardScaler()
scaler.fit(features)
standardized_data = scaler.transform(features)
features = standardized_data

---
## 5. Model Building & Prediction

Instead of importing a pre-built model, we build a **Logistic Regression** model from scratch using Gradient Descent to understand the underlying mathematics.

In [ ]:
class Logistic_Regression():
  def __init__(self, learning_rate, no_of_iterations):
    self.learning_rate = learning_rate
    self.no_of_iterations = no_of_iterations

  def fit(self, X, Y):
    # number of datapoints -> m, number of features -> n
    self.m, self.n = X.shape
    
    # initiating weights and bias
    self.w = np.zeros(self.n)
    self.b = 0
    self.X = X
    self.Y = Y
    
    # implementing gradient descent
    for i in range(self.no_of_iterations):
      self.update_weight()

  def update_weight(self):
    # calculating probability using sigmoid function
    y_hat = 1 / (1 + np.exp(-(self.X.dot(self.w) + self.b)))

    # calculating derivatives
    dw = (1 / self.m) * np.dot(self.X.T, (y_hat - self.Y))
    db = (1 / self.m) * np.sum(y_hat - self.Y)

    # updating weights
    self.w = self.w - self.learning_rate * dw
    self.b = self.b - self.learning_rate * db

  def predict(self, X):
    # mapping predicted values to classes (0 or 1)
    y_prediction = 1 / (1 + np.exp(-(X.dot(self.w) + self.b)))
    y_prediction = np.where(y_prediction > 0.5, 1, 0)
    return y_prediction

In [ ]:
# Split the data into Training and Testing sets
X_train, X_test, Y_train, Y_test = train_test_split(features, target, test_size=0.2, random_state=2)

# Initialize and train the custom model
classifier = Logistic_Regression(learning_rate=0.01, no_of_iterations=1000)
classifier.fit(X_train, Y_train)

# Accuracy on training data
X_training_prediction = classifier.predict(X_train)
training_data_accuracy = accuracy_score(Y_train, X_training_prediction)
print('Accuracy on Training Data:', training_data_accuracy)

# Accuracy on testing data
X_testing_prediction = classifier.predict(X_test)
testing_data_accuracy = accuracy_score(Y_test, X_testing_prediction)
print('Accuracy on Testing Data:', testing_data_accuracy)

### 5.3 Building a Predictive System

We create a helper function that takes live patient data, shapes and scales it, and passes it to our trained model for a diagnosis.

In [ ]:
def predict_diabetes(input_data):
    # Convert to numpy array and reshape for a single instance
    input_data_as_numpy_array = np.asarray(input_data)
    input_data_reshaped = input_data_as_numpy_array.reshape(1, -1)

    # Standardize the input data using the pre-fitted scaler
    std_data = scaler.transform(input_data_reshaped)

    # Make the prediction
    prediction = classifier.predict(std_data)
    print("Model Prediction Array:", prediction)

    if (prediction[0] == 0):
      print('Result: The person is NOT diabetic.')
    else:
      print('Result: The person IS diabetic.')

# Example usage (Passing 8 medical features)
predict_diabetes((1, 166, 50, 19, 18, 25.8, 0.587, 51))

---
## 6. Summary & Conclusion

### Key Findings

| Insight | Observation |
|---------|-------------|
| **Dataset size** | 768 rows × 9 columns |
| **Data Balance** | Mean outcome is 0.34 (Roughly 34% of the dataset is diabetic) |
| **Model Algorithm** | Logistic Regression (Built from scratch using math & numpy) |
| **Training Accuracy** | ~77.6% |
| **Testing Accuracy** | ~76.6% |

### Observations
- The custom model successfully converges, proving that building mathematical algorithms from scratch can rival standard library performance.
- Because the testing accuracy and training accuracy are nearly identical (~77% vs ~76%), the model generalizes well and **does not suffer from overfitting**.
- Medical features vary wildly in scale (e.g., Pedigree Function is < 1, while Insulin can be > 800). The `StandardScaler` was a vital step for Gradient Descent to work properly.

### Future Scope
- **UI Integration:** Connect this model to a Streamlit frontend web application for user-friendly testing.
- **Hyperparameter Tuning:** Test different learning rates (`0.1`, `0.001`) and iteration counts to see if accuracy improves.
- **Model Comparison:** Import `LogisticRegression` from `sklearn` and compare its accuracy directly against this custom-built implementation.

---
*Analysis performed using Python (Pandas, NumPy, Scikit-Learn)*